<a href="https://colab.research.google.com/github/amiralito/CarbonA_colab/blob/main/CarbonA_gene_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Carbon-A — batch gene prediction + BUSCO on Google Colab

[Carbon-A-1.2B](https://huggingface.co/HuggingFaceBio/Carbon-A-1.2B) (Hugging Face Biology) is a 1.2-billion-parameter bidirectional DNA language model trained as a eukaryotic **CDS annotator**: from genomic DNA alone it predicts, for **every base on both strands**, the probability that the base is protein-coding (non-overlapping 6-mer tokens, 98,304-bp context, trained on RefSeq assemblies across mammals, other vertebrates, invertebrates, plants, fungi and protozoa). This notebook runs it over **one or many genomes**, decodes the probability tracks into **gene models** (GFF3 + CDS + protein FASTA), writes **bigWig coding-probability tracks** for genome browsers, then scores completeness with **BUSCO** and compiles the results into a table.

> **About the gene models.** The released model stops at per-base CDS probabilities. The [technical report](https://huggingface.co/buckets/HuggingFaceBio/Carbon-A-Technical-Report) builds gene models with a learned *transition network* + a gene-grammar *Viterbi* decoder + a learned *confidence network*; only the annotator itself is public. Cell 8 re-implements the Viterbi stage as the report describes it (gene = ATG ... first in-frame stop; GT-AG / GC-AG / AT-AC introns with frequency log-priors; reading frame carried across splice junctions; emissions log *p* / log(1 − *p*)), with fixed, tunable boundary penalties in place of the transition network and a probability-based gene score in place of the confidence score. The gene models therefore approximate the report's pipeline rather than reproduce it. As in the report: one isoform per locus, CDS only (no UTRs).

**How to use:** set a GPU runtime (*Runtime > Change runtime type > GPU*). **A100 / L4 / H100 (bf16) are strongly recommended**; a free T4 works in fp16 but is several times slower — cell 6 measures the speed on your GPU and estimates the run time for your genomes. Fill the form fields and run each cell top to bottom. **The code is hidden** — to edit a cell, open its **menu (vertical dots) > Show code**.

Mount Drive in cell 3 to read genomes from a folder, cache the weights (4.6 GB, downloaded once) and keep results across sessions. Predictions are **checkpointed every few minutes**, so a long genome interrupted by a disconnect resumes where it stopped (`RESUME_RUN`, cell 3). Outputs are named after each genome (`<genome>_gff.gff3.gz`, `<genome>_cds.fasta.gz`, `<genome>_protein.fasta.gz`, `<genome>_cds_prob_{plus,minus}.bw`); gene IDs take the form `<PREFIX>_<seqid>_000001` (default `PREFIX = CarbonA`).

In [ ]:
#@title 1 · Check runtime — GPU { display-mode: "form" }
import sys, subprocess
print("Python:", sys.version.split()[0])
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    smi = ""
if smi:
    first = smi.splitlines()[0]
    print("GPU   :", first)
    GPU_NAME = first.split(",")[0].strip()
    GPU_MEM_GB = float(first.split(",")[1].strip().split()[0]) / 1024
    GPU_CC = float(first.split(",")[2])
    if GPU_CC >= 8.0:
        print("bf16-capable GPU: Carbon-A runs in bfloat16, as in the model's own inference checks.")
    else:
        print("Pre-Ampere GPU (e.g. T4): no bf16 - Carbon-A runs in float16 (cell 6 checks it against float32).\n"
              "Expect it to be several times slower than an A100 / L4.")
else:
    GPU_NAME, GPU_MEM_GB, GPU_CC = "", 0.0, 0.0
    print("No GPU. Carbon-A on a Colab CPU needs ~10 min per 100 kb - only for tiny tests.\n"
          "Runtime > Change runtime type > GPU.")

In [ ]:
#@title 2 · Install dependencies { display-mode: "form" }
#@markdown Carbon-A runs in Colab's own PyTorch through `transformers` (its custom model code is loaded with
#@markdown `trust_remote_code` from the downloaded, commit-pinned snapshot). This cell upgrades `transformers` only if it
#@markdown is older than 4.56, adds `pyBigWig` for the probability tracks, and writes `carbon_core.py` (6-mer encoder,
#@markdown window scheduler, gene-grammar Viterbi decoder, GFF3 writer) to `/content`. ~1 min; no restart needed.
import os, sys, subprocess, importlib
from importlib.metadata import version, PackageNotFoundError

def ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None

from packaging.version import Version
need = []
if ver("transformers") is None or Version(ver("transformers")) < Version("4.56"):
    need.append("transformers>=4.56,<6")
for pkg, spec in [("huggingface_hub", "huggingface_hub>=0.34"), ("safetensors", "safetensors>=0.4.3"),
                  ("biopython", "biopython>=1.83"), ("numba", "numba"), ("pyBigWig", "pyBigWig")]:
    if ver(pkg) is None:
        need.append(spec)
if need:
    print("$ pip install", " ".join(need))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

CORE = r'''# carbon_core.py - helpers for running Carbon-A and decoding its per-base CDS probabilities
# into gene models (GFF3 + CDS + protein). The decoder follows the gene grammar of the
# Carbon-A technical report (Section 5.5.3): a gene starts at ATG and ends at the first
# in-frame stop; introns run donor -> compatible acceptor (GT-AG, GC-AG, AT-AC) with the
# reading frame carried across the junction; emissions are log p (coding) / log(1-p)
# (non-coding); splice classes get fixed log-priors. The report's learned transition
# network and confidence network are NOT public, so boundaries here get fixed, tunable
# log-penalties and genes get a simple probability-based score instead.
import math
import numpy as np
import numba as nb

# ----------------------------------------------------------------------------- tokenizer
# Carbon-A vocab: 32 special tokens, then all 6-mers over "ATCG" in itertools.product order.
_LUT = np.full(256, 255, np.uint8)
for _i, _c in enumerate("ATCG"):
    _LUT[ord(_c)] = _i
    _LUT[ord(_c.lower())] = _i


def encode_6mers(seq_bytes):
    # seq_bytes: uint8 array (ASCII), length multiple of 6 -> int64 token ids (0 = <oov>)
    b = _LUT[seq_bytes].reshape(-1, 6)
    bad = (b == 255).any(1)
    w = np.array([4 ** 5, 4 ** 4, 4 ** 3, 4 ** 2, 4, 1], np.int64)
    ids = 32 + (b.astype(np.int64) * w).sum(1)
    ids[bad] = 0
    return ids


def window_starts(L, window, overlap):
    # paper: fixed windows with `overlap` bp shared, final window right-aligned to the record end
    if L <= window:
        return [0]
    stride = window - overlap
    starts = list(range(0, L - window + 1, stride))
    if starts[-1] + window < L:
        starts.append(L - window)
    return starts


# ----------------------------------------------------------------------------- decoder
# base codes for the decoder: A=0 C=1 G=2 T=3 other=4
_DLUT = np.full(256, 4, np.uint8)
for _i, _c in enumerate("ACGT"):
    _DLUT[ord(_c)] = _i
    _DLUT[ord(_c.lower())] = _i
_COMP = np.array([3, 2, 1, 0, 4], np.uint8)

A_, C_, G_, T_ = 0, 1, 2, 3
# states
IG = 0
E0, E1T, E1N, E2TA, E2TG, E2O = 1, 2, 3, 4, 5, 6
# intron states: 7 + t*6 + k ; k: 0 I0, 1 I1T, 2 I1N, 3 I2TA, 4 I2TG, 5 I2O ; t: 0 = GT/GC..AG, 1 = AT..AC
# start-codon states (A, T, G of the initiating ATG, which may not be split by an intron)
ST0, ST1, ST2 = 19, 20, 21
NS = 22
NEG = -1e300
SKIP = 100      # backpointer offset for "entered intron from exon state at j - MIN_INTRON"
START = 255


@nb.njit(cache=True, inline="always")
def _isstop(y, b, s_taa, s_tag, s_tga):
    # y: E2 prefix state, b: third base
    if y == E2TA:
        return (b == A_ and s_taa) or (b == G_ and s_tag)
    if y == E2TG:
        return b == A_ and s_tga
    return False


@nb.njit(cache=True)
def _forward(x, p, j0, j1, cur, ring, cum_in, store, bp, mini, gpen, ipen,
             r_gt, r_gc, r_at, s_taa, s_tag, s_tga):
    # Advance the Viterbi recursion over positions j0..j1-1. `cur` holds scores at j0-1
    # (ignored when j0 == 0). `ring` holds, for the last `mini` positions, exon-state scores
    # minus the running non-coding log-sum (for the minimum-length intron jump).
    L = x.shape[0]
    nxt = np.empty(NS, np.float64)
    cum = cum_in
    for j in range(j0, j1):
        pj = p[j]
        if pj != pj:            # NaN: unscored 6-mer -> neutral
            pj = 0.5
        if pj < 1e-6:
            pj = 1e-6
        elif pj > 1 - 1e-6:
            pj = 1 - 1e-6
        en = math.log(1.0 - pj)
        ec = math.log(pj) if x[j] < 4 else NEG
        cum += en
        if j == 0:
            for s in range(NS):
                nxt[s] = NEG
                if store:
                    bp[j - j0, s] = START
            nxt[IG] = en
            nxt[E0] = -gpen + ec
            nxt[E1N] = -gpen + ec
            nxt[E2O] = -gpen + ec
            nxt[7 + 0] = -gpen + en
            nxt[7 + 2] = -gpen + en
            nxt[7 + 5] = -gpen + en
        else:
            b = x[j - 1]
            # ---- intergenic: stay, or a gene ends after a completed stop codon at j-1
            best = cur[IG]; arg = IG
            for y in (E2TA, E2TG):
                if _isstop(y, b, s_taa, s_tag, s_tga) and cur[y] > best:
                    best = cur[y]; arg = y
            nxt[IG] = best + en
            if store:
                bp[j - j0, IG] = arg
            # acceptor checks for intron exit at j-1 (last two intron bases j-2, j-1)
            acc_ag = j >= 2 and x[j - 2] == A_ and b == G_
            acc_ac = j >= 2 and x[j - 2] == A_ and b == C_
            # ---- E0: gene start (ATG at j..j+2), next codon after a non-stop codon, or intron I0 exit
            best = NEG; arg = START
            if cur[ST2] > best:
                best = cur[ST2]; arg = ST2
            for y in (E2TA, E2TG, E2O):
                if not _isstop(y, b, s_taa, s_tag, s_tga) and cur[y] > best:
                    best = cur[y]; arg = y
            if acc_ag and cur[7] > best:
                best = cur[7]; arg = 7
            if acc_ac and cur[13] > best:
                best = cur[13]; arg = 13
            nxt[E0] = best + ec
            if store:
                bp[j - j0, E0] = arg
            # ---- E1T / E1N: second codon base; prefix = first base (at j-1 if contiguous)
            for tgt, k, want_t in ((E1T, 1, True), (E1N, 2, False)):
                best = NEG; arg = START
                if (b == T_) == want_t and cur[E0] > best:
                    best = cur[E0]; arg = E0
                if acc_ag and cur[7 + k] > best:
                    best = cur[7 + k]; arg = 7 + k
                if acc_ac and cur[13 + k] > best:
                    best = cur[13 + k]; arg = 13 + k
                nxt[tgt] = best + ec
                if store:
                    bp[j - j0, tgt] = arg
            # ---- E2*: third codon base; prefix class from E1 state + base at j-1
            for tgt, k in ((E2TA, 3), (E2TG, 4), (E2O, 5)):
                best = NEG; arg = START
                if tgt == E2TA:
                    if b == A_ and cur[E1T] > best:
                        best = cur[E1T]; arg = E1T
                elif tgt == E2TG:
                    if b == G_ and cur[E1T] > best:
                        best = cur[E1T]; arg = E1T
                else:
                    if b != A_ and b != G_ and cur[E1T] > best:
                        best = cur[E1T]; arg = E1T
                    if cur[E1N] > best:
                        best = cur[E1N]; arg = E1N
                if acc_ag and cur[7 + k] > best:
                    best = cur[7 + k]; arg = 7 + k
                if acc_ac and cur[13 + k] > best:
                    best = cur[13 + k]; arg = 13 + k
                nxt[tgt] = best + ec
                if store:
                    bp[j - j0, tgt] = arg
            # ---- start codon: IG -> A -> T -> G (only where the genome reads ATG at j..j+2)
            v = NEG
            if j + 2 < L and x[j] == A_ and x[j + 1] == T_ and x[j + 2] == G_:
                v = cur[IG] - gpen
            nxt[ST0] = v + ec
            nxt[ST1] = cur[ST0] + ec
            nxt[ST2] = cur[ST1] + ec
            if store:
                bp[j - j0, ST0] = IG
                bp[j - j0, ST1] = ST0
                bp[j - j0, ST2] = ST1
            # ---- introns: stay, or jump in from an exon state at i = j - mini
            i = j - mini
            slot = j % mini
            for t in range(2):
                base_s = 7 + 6 * t
                rho = NEG
                if i >= 0 and i + 2 < L:
                    d1 = x[i + 1]; d2 = x[i + 2]
                    if t == 0:
                        if d1 == G_ and d2 == T_:
                            rho = r_gt
                        elif d1 == G_ and d2 == C_:
                            rho = r_gc
                    else:
                        if d1 == A_ and d2 == T_:
                            rho = r_at
                for k in range(6):
                    s = base_s + k
                    best = cur[s] + en; arg = s
                    if rho > NEG:
                        xi = x[i]
                        src = -1
                        # which exon state at i feeds intron k, and with which emitted base
                        if k == 0:
                            # I0 <- E2y at i that completed a non-stop codon (pick best of the three)
                            for y in (E2TA, E2TG, E2O):
                                if not _isstop(y, xi, s_taa, s_tag, s_tga):
                                    v = ring[slot, y - 1] + cum + rho - ipen
                                    if v > best:
                                        best = v; arg = SKIP + y
                            v = ring[slot, 6] + cum + rho - ipen
                            if v > best:
                                best = v; arg = SKIP + ST2
                        elif k == 1:
                            if xi == T_:
                                src = E0
                        elif k == 2:
                            if xi != T_:
                                src = E0
                        elif k == 3:
                            if xi == A_:
                                src = E1T
                        elif k == 4:
                            if xi == G_:
                                src = E1T
                        else:
                            v = ring[slot, E1N - 1] + cum + rho - ipen
                            if v > best:
                                best = v; arg = SKIP + E1N
                            if xi != A_ and xi != G_:
                                src = E1T
                        if src >= 0:
                            v = ring[slot, src - 1] + cum + rho - ipen
                            if v > best:
                                best = v; arg = SKIP + src
                    nxt[s] = best
                    if store:
                        bp[j - j0, s] = arg
        # ring stores exon-state scores at j minus the non-coding log-sum through j
        slot = j % mini
        for e in range(6):
            ring[slot, e] = nxt[1 + e] - cum
        ring[slot, 6] = nxt[ST2] - cum
        for s in range(NS):
            cur[s] = nxt[s]
    return cum


@nb.njit(cache=True)
def _traceback_block(bp, j0, pos, state, mini, path):
    # walk back from (pos, state) while pos >= j0; returns (pos, state) to continue in the previous block
    while pos >= j0:
        cls = 0 if state == IG else (2 if 7 <= state <= 18 else 1)
        path[pos] = cls
        code = bp[pos - j0, state]
        if code == START:
            return -1, state
        if code >= SKIP:
            # positions pos-mini+1 .. pos are intron; exon state (code - SKIP) at pos - mini
            for q in range(pos - mini + 1, pos + 1):
                path[q] = 2
            pos = pos - mini
            state = code - SKIP
        else:
            pos -= 1
            state = code
    return pos, state


def viterbi_strand(x, p, min_intron=20, gene_penalty=8.0, intron_penalty=4.0,
                   splice_logprior=(math.log(0.9935), math.log(0.0056), math.log(0.0009)),
                   stops=(True, True, True), block=1 << 16):
    # x: uint8 base codes (A0 C1 G2 T3 other4) in transcript orientation; p: float32 coding prob.
    # Returns uint8 path (0 intergenic, 1 CDS, 2 intron). Memory: O(L) for the path,
    # O(block * 19) for backpointers (checkpointed Viterbi, the forward pass is run twice).
    L = x.shape[0]
    mini = max(int(min_intron), 4)
    r_gt, r_gc, r_at = splice_logprior
    s_taa, s_tag, s_tga = stops
    nblk = (L + block - 1) // block
    ck_cur = np.empty((nblk, NS)); ck_ring = np.empty((nblk, mini, 7)); ck_cum = np.empty(nblk)
    cur = np.full(NS, NEG); ring = np.full((mini, 7), NEG); cum = 0.0
    dummy = np.empty((1, NS), np.uint8)
    args = (mini, gene_penalty, intron_penalty, r_gt, r_gc, r_at, s_taa, s_tag, s_tga)
    for k in range(nblk):
        ck_cur[k] = cur; ck_ring[k] = ring; ck_cum[k] = cum
        cum = _forward(x, p, k * block, min(L, (k + 1) * block), cur, ring, cum, False, dummy, *args)
    path = np.zeros(L, np.uint8)
    pos, state = L - 1, int(np.argmax(cur))
    first_state = -1
    bp = np.empty((block, NS), np.uint8)
    for k in range(nblk - 1, -1, -1):
        if pos < 0:
            break
        j0 = k * block; j1 = min(L, (k + 1) * block)
        if pos >= j1:
            continue
        c = ck_cur[k].copy(); r = ck_ring[k].copy()
        _forward(x, p, j0, j1, c, r, ck_cum[k], True, bp, *args)
        pos, state = _traceback_block(bp, j0, pos, state, mini, path)
        if pos < 0:
            first_state = state
    return path, first_state


# ----------------------------------------------------------------------------- genes
STOPS_BY_CODE = {   # (TAA, TAG, TGA) are stops?
    1: (True, True, True), 6: (False, False, True), 10: (True, True, False), 12: (True, True, True),
    15: (True, False, True), 26: (True, True, True), 29: (False, False, True), 30: (False, False, True),
}


def _runs(mask):
    # [start, end) runs of True
    d = np.diff(np.concatenate(([0], mask.view(np.int8), [0])))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)


def genes_from_path(path, p, x):
    # path/p/x in transcript orientation. Returns list of dicts with exon intervals (transcript coords, half-open).
    L = path.shape[0]
    gs, ge = _runs(path != 0)
    out = []
    for s, e in zip(gs, ge):
        sub = path[s:e]
        es, ee = _runs(sub == 1)
        if len(es) == 0:
            continue
        exons = [(s + a, s + b) for a, b in zip(es, ee)]
        cds_len = sum(b - a for a, b in exons)
        pc = np.concatenate([p[a:b] for a, b in exons])
        pc = pc[~np.isnan(pc)]
        intr = [(exons[i][1], exons[i + 1][0]) for i in range(len(exons) - 1)]
        pi = np.concatenate([p[a:b] for a, b in intr]) if intr else np.array([], np.float32)
        pi = pi[~np.isnan(pi)]
        five = s > 0 and path[s - 1] == 0 and exons[0][0] == s
        three = e < L
        out.append(dict(exons=exons, cds_len=int(cds_len),
                        cds_mean=float(pc.mean()) if pc.size else 0.0,
                        intron_mean=float(1 - pi.mean()) if pi.size else 1.0,
                        partial5=not five, partial3=not three))
    return out


def genes_from_path_phase(path, first_state, p):
    # like genes_from_path, plus `offset` = codon position of the first CDS base for 5'-partial genes
    genes = genes_from_path(path, p, None)
    if genes and genes[0]["exons"][0][0] >= 0 and path[0] != 0:
        st = first_state
        if st in (E1T, E1N) or st in (8, 9, 14, 15):
            off = 1
        elif st in (E2TA, E2TG, E2O) or st in (10, 11, 12, 16, 17, 18):
            off = 2
        else:
            off = 0
        genes[0]["offset"] = off
    for g in genes:
        g.setdefault("offset", 0)
    return genes


_RC = bytes.maketrans(b"ACGTNacgtnRYKMSWBDHVrykmswbdhv", b"TGCANtgcanYRMKSWVHDByrmkswvhdb")


def annotate_record(seq_upper, p_plus, p_minus, min_intron=20, gene_penalty=8.0, intron_penalty=4.0,
                    genetic_code=1, block=1 << 16):
    # seq_upper: bytes (uppercase). Returns genes in genomic coordinates (0-based, half-open exons),
    # each with strand, exons (genomic order), cds (spliced, transcript orientation, bytes), phase offset.
    L = len(seq_upper)
    stops = STOPS_BY_CODE[int(genetic_code)]
    sb = np.frombuffer(seq_upper, np.uint8)
    x = _DLUT[sb]
    out = []
    for strand in "+-":
        if strand == "+":
            xs, ps = x, np.ascontiguousarray(p_plus, dtype=np.float32)
        else:
            xs = np.ascontiguousarray(_COMP[x[::-1]]); ps = np.ascontiguousarray(p_minus[::-1], dtype=np.float32)
        path, first = viterbi_strand(xs, ps, min_intron=min_intron, gene_penalty=gene_penalty,
                                     intron_penalty=intron_penalty, stops=stops, block=block)
        genes = genes_from_path_phase(path, first, ps)
        for g in genes:
            ex_t = g["exons"]
            if strand == "+":
                ex_g = [(int(a), int(b)) for a, b in ex_t]
                cds = b"".join(seq_upper[a:b] for a, b in ex_g)
            else:
                ex_g = [(int(L - b), int(L - a)) for a, b in ex_t][::-1]
                cds = b"".join(seq_upper[a:b].translate(_RC)[::-1] for a, b in reversed(ex_g))
            g.update(strand=strand, exons=ex_g, cds=cds)
            out.append(g)
        del path
    out.sort(key=lambda g: (g["exons"][0][0], g["strand"]))
    return out


def translate_cds(cds, offset, genetic_code, partial3):
    from Bio.Seq import Seq
    s = cds[(3 - offset) % 3:]
    s = s[: len(s) - len(s) % 3]
    aa = str(Seq(s.decode()).translate(table=int(genetic_code)))
    if not partial3 and aa.endswith("*"):
        aa = aa[:-1]
    return aa


def gff_lines(seqid, genes, prefix, source="Carbon-A", start_index=1):
    # GFF3 gene/mRNA/exon/CDS lines (no UTRs: Carbon-A predicts CDS only)
    lines = []
    for n, g in enumerate(genes, start_index):
        gid = f"{prefix}_{seqid}_{n:06d}"; tid = gid + ".1"
        g["gene_id"], g["transcript_id"] = gid, tid
        st = g["exons"][0][0] + 1; en = g["exons"][-1][1]
        sc = f"{g['score']:.3f}"
        part = []
        if g["partial5"]: part.append("5prime")
        if g["partial3"]: part.append("3prime")
        extra = f";mean_cds_p={g['cds_mean']:.3f};mean_intron_q={g['intron_mean']:.3f}"
        if part:
            extra += ";partial=" + ",".join(part)
        lines.append(f"{seqid}\t{source}\tgene\t{st}\t{en}\t{sc}\t{g['strand']}\t.\tID={gid}{extra}")
        lines.append(f"{seqid}\t{source}\tmRNA\t{st}\t{en}\t{sc}\t{g['strand']}\t.\tID={tid};Parent={gid}")
        tx = g["exons"] if g["strand"] == "+" else g["exons"][::-1]       # transcript order
        cum = g["offset"]
        rows = []
        for k, (a, b) in enumerate(tx, 1):
            ph = (3 - cum % 3) % 3
            rows.append((a, b, k, ph))
            cum += b - a
        rows.sort()
        for a, b, k, ph in rows:
            lines.append(f"{seqid}\t{source}\texon\t{a + 1}\t{b}\t.\t{g['strand']}\t.\tID={tid}.exon.{k};Parent={tid}")
        for a, b, k, ph in rows:
            lines.append(f"{seqid}\t{source}\tCDS\t{a + 1}\t{b}\t.\t{g['strand']}\t{ph}\tID={tid}.CDS.{k};Parent={tid}")
    return lines


# ----------------------------------------------------------------------------- I/O + inference
def read_fasta(path, with_seq=True):
    # streams (header, sequence bytes uppercased) - or (header, length) with with_seq=False
    import gzip
    op = gzip.open if str(path).endswith(".gz") else open
    head, chunks, n = None, [], 0
    with op(path, "rb") as fh:
        for line in fh:
            if line.startswith(b">"):
                if head is not None:
                    yield head, (b"".join(chunks).upper() if with_seq else n)
                head, chunks, n = line[1:].decode(errors="replace").strip(), [], 0
            else:
                line = line.strip()
                if with_seq:
                    chunks.append(line)
                else:
                    n += len(line)
        if head is not None:
            yield head, (b"".join(chunks).upper() if with_seq else n)


def seqid_of(header):
    return header.split()[0] if header.split() else "unnamed"


QSCALE = 8.0    # stored logits are int8 in 1/8-nat steps (|logit| <= 15.875); -128 = unscored


def quantize_logits(d):
    q = np.clip(np.rint(np.nan_to_num(d) * QSCALE), -127, 127).astype(np.int8)
    q[np.isnan(d)] = -128
    return q


def predict_windows(model, seq_upper, starts, window, device):
    # One forward pass over windows of equal length. Returns int8 quantized logit(p) arrays of shape
    # (2, w) per window (row 0 = forward strand, row 1 = reverse strand, both in genomic coordinates),
    # -128 where the 6-mer contains a non-ACGT base (or is N padding).
    import torch
    frags = []
    for s in starts:
        f = seq_upper[s:s + window]
        frags.append(f + b"N" * (-len(f) % 6))
    assert len({len(f) for f in frags}) == 1, "windows in one batch must have equal length"
    ids = np.stack([encode_6mers(np.frombuffer(f, np.uint8)) for f in frags])
    tid = torch.from_numpy(ids).to(device)
    mask = torch.ones_like(tid)
    with torch.inference_mode():
        if tid.is_cuda:
            from torch.nn.attention import sdpa_kernel, SDPBackend
            # fused kernels only: the plain "math" fallback would need >32 GB for a full window
            with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):
                lg = model(input_ids=tid, attention_mask=mask).logits
        else:
            lg = model(input_ids=tid, attention_mask=mask).logits
        d = lg[..., 1].float() - lg[..., 0].float()
        if not bool(torch.isfinite(d).all()):
            raise FloatingPointError("non-finite model output - set PRECISION = fp32 in cell 4")
        d = d.reshape(len(frags), 2, -1).cpu().numpy()
    bad = np.repeat(ids == 0, 6, axis=1)
    d[np.broadcast_to(bad[:, None, :], d.shape)] = np.nan
    out = []
    for k, s in enumerate(starts):
        w = min(window, len(seq_upper) - s)
        out.append(quantize_logits(d[k, :, :w]))
    return out


def accumulate(L, windows):
    # windows: iterable of (start, int8 logits (2, w)); returns averaged probabilities (2, L) float32, NaN = unscored
    sums = np.zeros((2, L), np.float32)
    cnt = np.zeros(L, np.uint8)
    for s, q in windows:
        w = q.shape[1]
        v = q[0] != -128
        pr = 1.0 / (1.0 + np.exp(-q.astype(np.float32) / QSCALE))
        sums[:, s:s + w] += np.where(v, pr, 0)
        cnt[s:s + w] += v
    ok = cnt > 0
    sums[:, ok] /= cnt[ok]
    sums[:, ~ok] = np.nan
    return sums
'''
with open("/content/carbon_core.py", "w") as fh:
    fh.write(CORE)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import carbon_core
importlib.reload(carbon_core)

# compile the decoder once now (numba), so cell 8 starts straight away
import numpy as np
_x = np.random.default_rng(0).integers(0, 4, 5000).astype(np.uint8)
carbon_core.viterbi_strand(_x, np.full(5000, 0.1, np.float32), block=1024)

import torch, transformers, numba, Bio, pyBigWig
for name, mod in [("torch", torch), ("transformers", transformers), ("numba", numba), ("biopython", Bio)]:
    print(f"{name:<13}: {mod.__version__}")
print(f"{'pyBigWig':<13}: {ver('pyBigWig')} (numpy support: {bool(pyBigWig.numpy)})")
print(f"{'CUDA (torch)':<13}: {torch.version.cuda}  | GPU visible: {torch.cuda.is_available()}")
if Version(torch.__version__.split('+')[0]) < Version("2.7"):
    print("WARNING: Carbon-A's requirements ask for torch >= 2.7; this runtime has", torch.__version__)
print("\nDone. carbon_core.py written to /content.")

In [ ]:
#@title 3 · Storage — mount Drive & start a run folder { display-mode: "form" }
#@markdown Mount Google Drive so inputs, the weights and outputs persist across sessions.
#@markdown Every run gets its own timestamped folder under `WORKDIR/runs/`, so nothing is overwritten.
MOUNT_DRIVE = True  #@param {type:"boolean"}
WORKDIR = "/content/drive/MyDrive/carbonA_runs"  #@param {type:"string"}
RUN_TAG = ""  #@param {type:"string"}
TIMEZONE = "Europe/London"  #@param {type:"string"}
#@markdown `RUN_TAG` is an optional suffix for the folder name (e.g. `oomycetes`). `TIMEZONE` only
#@markdown affects the timestamp; the Colab clock itself is UTC.
RESUME_RUN = ""  #@param {type:"string"}
#@markdown To **continue an interrupted batch** in a new session, paste the existing run folder name here
#@markdown (e.g. `run_20261009-142530_oomycetes`) and run cells 1-7 again: finished genomes are skipped
#@markdown (`SKIP_DONE`, cell 4) and a genome that was cut off resumes from its last checkpoint.

import os, re
from datetime import datetime, timezone as _tz

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
elif WORKDIR.startswith("/content/drive"):
    WORKDIR = "/content/work"

try:
    from zoneinfo import ZoneInfo
    NOW = datetime.now(ZoneInfo(TIMEZONE))
except Exception:
    NOW, TIMEZONE = datetime.now(_tz.utc), "UTC"

tag = re.sub(r"[^A-Za-z0-9._-]+", "_", RUN_TAG.strip()).strip("_")
if RESUME_RUN.strip():
    RUN_ID = os.path.basename(RESUME_RUN.strip().rstrip("/"))
    assert os.path.isdir(os.path.join(WORKDIR, "runs", RUN_ID)), \
        f"No run folder {RUN_ID} under {os.path.join(WORKDIR, 'runs')}"
    print("Resuming existing run folder.")
else:
    RUN_ID = NOW.strftime("run_%Y%m%d-%H%M%S") + (f"_{tag}" if tag else "")
RUNDIR = os.path.join(WORKDIR, "runs", RUN_ID)
CKPT_ROOT = os.path.join(WORKDIR, "carbonA_model")     # shared across runs, downloaded once
os.makedirs(RUNDIR, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)

print("Working directory:", WORKDIR)
print("Run folder       :", RUNDIR, f"({TIMEZONE})")
print("Weights cache    :", CKPT_ROOT)
print("\nRe-run this cell to start a new run folder; re-running cells 7-13 alone keeps writing to this one.")

In [ ]:
#@title 4 · Prediction parameters & genome input { display-mode: "form" }
#@markdown **Model**
MODEL_REVISION = "main"  #@param {type:"string"}
#@markdown A branch or commit of `HuggingFaceBio/Carbon-A-1.2B`. It is resolved to a commit hash in cell 6, which
#@markdown names the cached weights folder and is recorded in `run_config.json` - paste that hash here to repeat a run exactly.
PRECISION = "auto"  #@param ["auto", "bf16", "fp16", "fp32"]
#@markdown `auto` = bf16 on A100 / L4 / H100 and newer, fp16 on T4 / V100, fp32 on CPU.
WINDOW_OVERLAP = "6144"  #@param ["6144", "49152"] {allow-input: true}
#@markdown Overlap between consecutive 98,304-bp windows; predictions are averaged where windows overlap.
#@markdown `6144` (1,024 tokens) is the technical report's genome-scale setting; `49152` (50%) is what the model card's
#@markdown `infer_genbank.py` uses and costs ~2x the GPU time. Must be a multiple of 6.
BATCH_SIZE = "auto"  #@param ["auto", "1", "2", "4", "8"] {allow-input: true}
#@markdown Full-length windows per forward pass. `auto` = 1 below 40 GB of GPU memory, 2 on 40 GB, 4 on 80 GB.
#@markdown Larger batches only help on large GPUs; on OOM the batch is halved automatically.
MIN_RECORD_LENGTH = 1000  #@param {type:"integer"}
#@markdown Sequences shorter than this are skipped (the report's pipeline skips < 100,000 bp; training used full
#@markdown 98,304-bp windows, so predictions on very short contigs are less reliable).
CHECKPOINT_MINUTES = 10  #@param {type:"number"}
#@markdown Window predictions are written to the run folder at least this often (resume granularity).
SKIP_DONE = True  #@param {type:"boolean"}
#@markdown `SKIP_DONE`: genomes already predicted in **this run folder** are not re-run (cell 7), and a genome cut off by
#@markdown a disconnect continues from its last checkpoint.
STOP_ON_ERROR = False  #@param {type:"boolean"}
#@markdown Off: a failed genome is logged and the batch moves on (failures are listed at the end of cell 7).
#@markdown ---
#@markdown **Genome input** — `drive_dir` runs on **every** FASTA found recursively under the folder; the others take a single genome.
GENOME_SOURCE = "drive_dir"  #@param ["drive_dir", "upload", "url", "drive_file"]
GENOME_DIR = "/content/drive/MyDrive/genomes"  #@param {type:"string"}
GENOME_PATH = "/content/drive/MyDrive/genome.fasta"  #@param {type:"string"}
GENOME_URL = ""  #@param {type:"string"}

WINDOW = 98304                                   # 16,384 6-mer tokens, the model's full context
OVERLAP_BP = int(WINDOW_OVERLAP)
assert 0 <= OVERLAP_BP < WINDOW and OVERLAP_BP % 6 == 0, "WINDOW_OVERLAP must be a multiple of 6 and < 98,304."
assert MIN_RECORD_LENGTH >= 1 and CHECKPOINT_MINUTES > 0
if str(BATCH_SIZE).strip() == "auto":
    BATCH = 4 if GPU_MEM_GB >= 70 else (2 if GPU_MEM_GB >= 35 else 1)
else:
    BATCH = max(1, int(BATCH_SIZE))

print(f"revision {MODEL_REVISION} | precision {PRECISION} | window {WINDOW} bp, overlap {OVERLAP_BP} bp "
      f"(stride {WINDOW - OVERLAP_BP}) | batch {BATCH} | min record {MIN_RECORD_LENGTH} bp | source {GENOME_SOURCE}")

In [ ]:
#@title 5 · Collect genome(s) { display-mode: "form" }
#@markdown Finds the FASTA file(s) and scans their sequence lengths (FASTA may be gzipped; soft-masked lowercase is fine -
#@markdown sequences are upper-cased before tokenization, as the model requires). Files on Drive are read in place.
import os, subprocess
import carbon_core as cc

STAGE = "/content/run"; GEN_DIR = os.path.join(STAGE, "genomes"); os.makedirs(GEN_DIR, exist_ok=True)
FASTA_EXT = (".fa", ".fasta", ".fna", ".fas", ".fa.gz", ".fasta.gz", ".fna.gz", ".fas.gz")

def base_of(p):
    b = os.path.basename(p)
    for e in sorted(FASTA_EXT, key=len, reverse=True):
        if b.lower().endswith(e):
            return b[:-len(e)]
    return os.path.splitext(b)[0]

raw = []
if GENOME_SOURCE == "drive_dir":
    assert MOUNT_DRIVE, "Enable MOUNT_DRIVE in cell 3 to read from Drive."
    assert os.path.isdir(GENOME_DIR), f"Not a directory: {GENOME_DIR}"
    for root, _, fnames in os.walk(GENOME_DIR):
        for fn in fnames:
            if fn.lower().endswith(FASTA_EXT):
                raw.append(os.path.join(root, fn))
    assert raw, f"No FASTA (.fa/.fasta/.fna/.fas[.gz]) found under {GENOME_DIR}"
elif GENOME_SOURCE == "drive_file":
    assert os.path.exists(GENOME_PATH), f"Not found: {GENOME_PATH}"
    raw = [GENOME_PATH]
elif GENOME_SOURCE == "url":
    assert GENOME_URL, "Set GENOME_URL in cell 4."
    dst = os.path.join(GEN_DIR, GENOME_URL.split("/")[-1].split("?")[0])
    subprocess.run(["wget", "-q", "-O", dst, GENOME_URL], check=True)
    raw = [dst]
elif GENOME_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    for name in up:
        d = os.path.join(GEN_DIR, name); os.replace(name, d); raw.append(d)

GENOMES = [(base_of(p), p) for p in sorted(raw)]
dupes = {b for b, _ in GENOMES if [x for x, _ in GENOMES].count(b) > 1}
assert not dupes, f"Genome names must be unique - these appear more than once: {sorted(dupes)}"

GENOME_INFO = {}           # base -> list of (seqid, length) in file order
print(f"{'genome':<40}{'records':>9}{'Mb':>10}{'kept':>8}{'kept Mb':>10}{'windows':>9}")
for b, p in GENOMES:
    recs = [(cc.seqid_of(h), n) for h, n in cc.read_fasta(p, with_seq=False)]
    ids = [s for s, _ in recs]
    assert len(set(ids)) == len(ids), f"{b}: duplicate sequence IDs (first word of the FASTA headers)"
    assert recs, f"{b}: no FASTA records found"
    GENOME_INFO[b] = recs
    kept = [n for _, n in recs if n >= MIN_RECORD_LENGTH]
    nwin = sum(len(cc.window_starts(n, WINDOW, OVERLAP_BP)) for n in kept)
    print(f"{b:<40}{len(recs):>9}{sum(n for _, n in recs) / 1e6:>10.1f}{len(kept):>8}{sum(kept) / 1e6:>10.1f}{nwin:>9}")
print(f"\n{len(GENOMES)} genome(s); sequences < {MIN_RECORD_LENGTH} bp are skipped.")

In [ ]:
#@title 6 · Download Carbon-A & verify it on this GPU { display-mode: "form" }
#@markdown Downloads the weights (4.6 GB) once into `WORKDIR/carbonA_model/<commit>/` and loads them in the chosen precision.
#@markdown Checks: the notebook's 6-mer encoder against the model's own tokenizer; finite output on a full 98,304-bp window
#@markdown of your first genome, timed to **estimate the GPU time for the whole batch**; and, in fp16, agreement with fp32.
import os, glob, time, json, numpy as np, torch
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoModelForTokenClassification, AutoTokenizer
import carbon_core as cc

MODEL_ID = "HuggingFaceBio/Carbon-A-1.2B"
try:
    MODEL_SHA = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
except Exception as e:
    cached = sorted(glob.glob(os.path.join(CKPT_ROOT, "*", "model.safetensors")), key=os.path.getmtime)
    assert cached, f"Cannot reach the Hugging Face Hub ({e}) and no cached weights in {CKPT_ROOT}"
    MODEL_SHA = os.path.basename(os.path.dirname(cached[-1]))
    print(f"Hub unreachable ({type(e).__name__}); using cached weights {MODEL_SHA}")
MODEL_DIR = os.path.join(CKPT_ROOT, MODEL_SHA[:12])
weights = os.path.join(MODEL_DIR, "model.safetensors")
if not (os.path.exists(weights) and os.path.getsize(weights) > 4e9):
    print(f"downloading {MODEL_ID}@{MODEL_SHA[:12]} -> {MODEL_DIR}")
    snapshot_download(MODEL_ID, revision=MODEL_SHA, local_dir=MODEL_DIR)
else:
    print("weights already cached:", MODEL_DIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cap = torch.cuda.get_device_capability() if DEVICE == "cuda" else (0, 0)
PREC = PRECISION
if PREC == "auto":
    PREC = ("bf16" if cap[0] >= 8 else "fp16") if DEVICE == "cuda" else "fp32"
if DEVICE == "cpu" and PREC != "fp32":
    print("CPU runtime: using fp32."); PREC = "fp32"
if PREC == "bf16" and cap[0] < 8:
    print("WARNING: this GPU has no native bf16 (it would be emulated, slowly) - fp16 is the better choice here.")
DTYPE = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}[PREC]

if "MODEL" in globals():
    del MODEL
    torch.cuda.empty_cache()
MODEL = AutoModelForTokenClassification.from_pretrained(
    MODEL_DIR, trust_remote_code=True, dtype=DTYPE, attn_implementation="sdpa").to(DEVICE).eval()
c = MODEL.config
assert (c.k, c.num_prediction_heads, c.num_labels) == (6, 2, 2), "Expected a 6-mer, two-strand binary CDS model."
print(f"loaded {MODEL_ID}@{MODEL_SHA[:12]} | {sum(p.numel() for p in MODEL.parameters()) / 1e9:.2f} B params | {PREC} on {DEVICE}")

# 1) encoder == the model's tokenizer (uppercase, no special tokens, ambiguous bases -> <oov>)
tok = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
rng = np.random.default_rng(0)
probe = "".join(rng.choice(list("ACGTACGTACGTNRY"), 6 * 2000))
assert tok(probe, add_special_tokens=False)["input_ids"] == cc.encode_6mers(np.frombuffer(probe.encode(), np.uint8)).tolist(), \
    "6-mer encoder disagrees with the model tokenizer"
print("encoder check   : identical to the model's tokenizer")

# 2) a full-length window of real sequence: finite output + timing
first_base, first_path = GENOMES[0]
seq = None
for h, s in cc.read_fasta(first_path):
    if len(s) >= WINDOW or seq is None or len(s) > len(seq):
        seq = s
    if len(seq) >= WINDOW:
        break
test = seq[:WINDOW]
if len(test) < WINDOW:                       # tiny genome: tile it to a full window for the timing
    test = (test * (WINDOW // max(len(test), 1) + 1))[:WINDOW]

def run(bs):
    return cc.predict_windows(MODEL, test * bs, [i * WINDOW for i in range(bs)], WINDOW, DEVICE)

def timed(bs, reps=2):
    run(bs)                                   # warm-up
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        run(bs)
    if DEVICE == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / reps / bs

if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
try:
    t_win = timed(BATCH, reps=2 if DEVICE == "cuda" else 1)
except FloatingPointError as e:
    if PREC != "fp16":
        raise
    print("fp16 produced non-finite output on this sequence - switching to fp32.")
    MODEL.float(); PREC, DTYPE = "fp32", torch.float32
    t_win = timed(BATCH)
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    assert BATCH > 1, "Out of GPU memory with a single window - use a GPU with more memory."
    BATCH = max(1, BATCH // 2); print(f"OOM - batch reduced to {BATCH}")
    t_win = timed(BATCH)
peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE == "cuda" else 0
print(f"window check    : finite | {t_win:.2f} s per {WINDOW:,}-bp window (batch {BATCH})" +
      (f" | peak GPU memory {peak:.1f} GB" if peak else ""))

# 3) fp16 vs fp32 on 12,288 bp
if PREC == "fp16":
    sub = test[:12288]
    a = cc.predict_windows(MODEL, sub, [0], WINDOW, DEVICE)[0]
    MODEL.float()
    b = cc.predict_windows(MODEL, sub, [0], WINDOW, DEVICE)[0]
    MODEL.half()
    v = (a != -128) & (b != -128)
    pa = 1 / (1 + np.exp(-a[v] / cc.QSCALE)); pb = 1 / (1 + np.exp(-b[v] / cc.QSCALE))
    print(f"fp16 vs fp32    : max |dp| {np.abs(pa - pb).max():.3f} | 0.5-call agreement {((pa >= .5) == (pb >= .5)).mean() * 100:.2f}%")

# run-time estimate for the batch
tot = 0.0
for b, recs in GENOME_INFO.items():
    for _, n in recs:
        if n >= MIN_RECORD_LENGTH:
            nw = len(cc.window_starts(n, WINDOW, OVERLAP_BP))
            tot += t_win * (nw if n >= WINDOW else max(n / WINDOW, 0.02))
print(f"\nestimated GPU time for all {len(GENOME_INFO)} genome(s): ~{tot / 3600:.1f} h "
      f"(~{t_win * 1e8 / (WINDOW - OVERLAP_BP) / 60:.0f} min per 100 Mb)")
if tot > 10 * 3600:
    print("That exceeds a typical Colab session - predictions are checkpointed, so you can resume (RESUME_RUN, cell 3).")

In [ ]:
#@title 7 · Run Carbon-A on all genome(s) — GPU { display-mode: "form" }
#@markdown Each sequence is cut into 98,304-bp windows (the last one right-aligned to the sequence end) and Carbon-A predicts
#@markdown both strands. Window predictions are stored compactly (int8 logits, ~1 byte/bp) under
#@markdown `carbonA_predictions/windows/<genome>/` and flushed every `CHECKPOINT_MINUTES`; after a disconnect, set `RESUME_RUN`
#@markdown (cell 3), run cells 1-7 again and only the missing windows are predicted. Gene models are built in cell 8.
import os, json, glob, time, numpy as np, torch
from datetime import datetime
import carbon_core as cc

OUTBASE = os.path.join(RUNDIR, "carbonA_predictions")
WIN_ROOT = os.path.join(OUTBASE, "windows")
LOG_DIR = os.path.join(OUTBASE, "log")
for d in (WIN_ROOT, LOG_DIR):
    os.makedirs(d, exist_ok=True)

PRED_KEY = {"model": MODEL_ID, "revision": MODEL_SHA, "window": WINDOW, "overlap": OVERLAP_BP,
            "min_record_length": MIN_RECORD_LENGTH}

# a record of exactly what produced this run folder (a resumed run adds a second, timestamped record)
cfg_path = os.path.join(RUNDIR, "run_config.json")
if os.path.exists(cfg_path):
    cfg_path = os.path.join(RUNDIR, f"run_config_resumed_{datetime.now(NOW.tzinfo):%Y%m%d-%H%M%S}.json")
import transformers
with open(cfg_path, "w") as fh:
    json.dump({"run_id": RUN_ID, "started": NOW.isoformat(), "timezone": TIMEZONE, **PRED_KEY,
               "precision": PREC, "batch_size": BATCH, "checkpoint_minutes": CHECKPOINT_MINUTES,
               "genomes": {b: p for b, p in GENOMES},
               "environment": {"torch": torch.__version__, "transformers": transformers.__version__,
                               "cuda": torch.version.cuda, "gpu": GPU_NAME or "cpu"}}, fh, indent=2)

def flush(gdir, buf, k):
    # one chunk = many windows: record index, start, length + their concatenated int8 logits (2, sum(len))
    if not buf:
        return k
    tmp = os.path.join(gdir, f".chunk_{k:05d}.tmp.npz")
    np.savez_compressed(tmp, rec=np.array([b[0] for b in buf], np.int32),
                        start=np.array([b[1] for b in buf], np.int64),
                        length=np.array([b[2].shape[1] for b in buf], np.int64),
                        logits=np.concatenate([b[2] for b in buf], axis=1))
    os.replace(tmp, os.path.join(gdir, f"chunk_{k:05d}.npz"))
    return k + 1

PREDICTED, FAILED = [], []
for gi, (base, gpath) in enumerate(GENOMES, 1):
    print(f"\n===== [{gi}/{len(GENOMES)}] {base} =====")
    gdir = os.path.join(WIN_ROOT, base); os.makedirs(gdir, exist_ok=True)
    done_flag = os.path.join(gdir, "DONE.json")
    buf, k = [], 0
    logf = open(os.path.join(LOG_DIR, f"{base}.carbonA.log"), "a")
    def log(msg):
        print(msg); logf.write(f"{datetime.now(NOW.tzinfo):%Y-%m-%d %H:%M:%S}  {msg}\n"); logf.flush()
    try:
        if os.path.exists(done_flag):
            if SKIP_DONE:
                print("already predicted in this run folder - skipped (SKIP_DONE)")
                PREDICTED.append(base); continue
            for f in glob.glob(os.path.join(gdir, "*chunk_*.npz")) + [done_flag]:
                os.remove(f)
        man_path = os.path.join(gdir, "manifest.json")
        recs = GENOME_INFO[base]
        if os.path.exists(man_path):
            old = json.load(open(man_path))
            assert old["key"] == PRED_KEY, (f"{base}: this run folder holds predictions made with other settings "
                                            f"({old['key']}); start a new run folder (cell 3) or restore those settings.")
            assert [tuple(r) for r in old["records"]] == [tuple(r) for r in recs], \
                f"{base}: the FASTA differs from the one predicted in this run folder."
        else:
            with open(man_path, "w") as fh:
                json.dump({"key": PRED_KEY, "genome": gpath, "records": recs}, fh)
        done = set()
        chunks = sorted(glob.glob(os.path.join(gdir, "chunk_*.npz")))
        for f in chunks:
            with np.load(f) as z:
                done.update(zip(z["rec"].tolist(), z["start"].tolist()))
        k = (max(int(os.path.basename(f)[6:11]) for f in chunks) + 1) if chunks else 0
        todo_bp = sum(n for _, n in recs if n >= MIN_RECORD_LENGTH)
        if done:
            log(f"resuming: {len(done)} window(s) already predicted")
        t0 = last_flush = last_print = time.time()
        buf, buf_bytes, n_win, bp_done = [], 0, 0, 0
        for idx, (h, seq) in enumerate(cc.read_fasta(gpath)):
            L = len(seq)
            if L < MIN_RECORD_LENGTH:
                continue
            starts = cc.window_starts(L, WINDOW, OVERLAP_BP)
            todo = [s for s in starts if (idx, s) not in done]
            step = BATCH if L >= WINDOW else 1
            i = 0
            while i < len(todo):
                bs = todo[i:i + step]
                try:
                    outs = cc.predict_windows(MODEL, seq, bs, WINDOW, DEVICE)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if step == 1:
                        raise
                    step = BATCH = max(1, step // 2); log(f"GPU OOM - batch size reduced to {BATCH}")
                    continue
                for s, q in zip(bs, outs):
                    buf.append((idx, s, q)); buf_bytes += q.nbytes
                n_win += len(bs); i += len(bs)
                now = time.time()
                if now - last_flush > CHECKPOINT_MINUTES * 60 or buf_bytes > 3e8:
                    k = flush(gdir, buf, k); buf, buf_bytes, last_flush = [], 0, now
                if now - last_print > 120:
                    frac = (bp_done + (i / max(len(starts), 1)) * L) / todo_bp
                    rate = frac * todo_bp / 1e6 / ((now - t0) / 60)
                    log(f"  {frac * 100:5.1f}% | record {idx + 1}/{len(recs)} {cc.seqid_of(h)} | {n_win} windows | "
                        f"{rate:.2f} Mb/min | ETA {(1 - frac) * todo_bp / 1e6 / max(rate, 1e-9):.0f} min")
                    last_print = now
            bp_done += L
        k = flush(gdir, buf, k); buf = []
        with open(done_flag, "w") as fh:
            json.dump({"records_predicted": sum(1 for _, n in recs if n >= MIN_RECORD_LENGTH),
                       "bp_predicted": todo_bp, "windows_this_session": n_win,
                       "minutes_this_session": round((time.time() - t0) / 60, 1), "precision": PREC}, fh, indent=2)
        log(f"done: {todo_bp / 1e6:.1f} Mb, {n_win} new window(s) in {(time.time() - t0) / 60:.1f} min")
        PREDICTED.append(base)
    except Exception as e:
        try:
            flush(gdir, buf, k)          # keep what was predicted before the error
        except Exception:
            pass
        msg = f"Carbon-A failed on {base}: {type(e).__name__}: {e}"
        if STOP_ON_ERROR:
            raise
        log("ERROR: " + msg); FAILED.append(base)
    finally:
        logf.close()

print(f"\nDone. {len(PREDICTED)} genome(s) predicted under {WIN_ROOT}/")
if FAILED:
    print(f"FAILED ({len(FAILED)}):", ", ".join(FAILED), "- see their logs in", LOG_DIR)

In [ ]:
#@title 8 · Decode gene models → GFF3 + CDS + protein (+ bigWig tracks) { display-mode: "form" }
#@markdown Rebuilds per-base probabilities from the saved windows (overlaps averaged) and decodes each strand with the
#@markdown gene-grammar Viterbi (see the note at the top). CPU only, ~1-2 min per 100 Mb - **re-run this cell alone to try
#@markdown other settings**; the GPU step is not repeated.
PREFIX = "CarbonA"  #@param {type:"string"}
#@markdown Gene-ID prefix: `CarbonA_<seqid>_000001`, transcripts `.1`. Leave empty to use each genome's file name.
GENE_PENALTY = 8.0  #@param {type:"number"}
INTRON_PENALTY = 4.0  #@param {type:"number"}
#@markdown Log-penalties (nats) for opening a gene / an intron, standing in for the report's unreleased transition network.
#@markdown Higher values give fewer, longer genes / fewer introns. Coding evidence dominates: results are stable over ~0-8.
MIN_INTRON_LENGTH = 20  #@param {type:"integer"}
GENETIC_CODE = "1 · standard"  #@param ["1 · standard", "6 · ciliate (TAA/TAG = Gln)", "10 · euplotid (TGA = Cys)", "12 · alternative yeast (CTG = Ser)", "15 · Blepharisma (TAG = Gln)", "26 · Pachysolen (CTG = Ala)", "29 · Mesodinium (TAA/TAG = Tyr)", "30 · peritrich (TAA/TAG = Glu)"]
#@markdown Sets which codons end a gene and how CDS are translated (NCBI table number).
MIN_CDS_LENGTH = 96  #@param {type:"integer"}
#@markdown 96 bp is the report's default.
MIN_SCORE = 0.0  #@param {type:"number"}
#@markdown Gene score (GFF3 column 6) = mean CDS probability x mean non-coding probability over its introns, 0-1.
#@markdown 0 keeps everything; ~0.5 drops weakly supported models.
KEEP_PARTIAL = True  #@param {type:"boolean"}
#@markdown Genes running off a sequence end (missing start and/or stop) - flagged `partial=5prime/3prime`.
WRITE_BIGWIG = True  #@param {type:"boolean"}
#@markdown Per-base CDS probability per strand (`_cds_prob_plus.bw` / `_cds_prob_minus.bw`, 3 decimals) for IGV / JBrowse.
import os, re, json, glob, gzip, shutil, time, numpy as np
from datetime import datetime
import carbon_core as cc

CODE = int(GENETIC_CODE.split()[0])
PRED_LOCAL = "/content/run/pred"; os.makedirs(PRED_LOCAL, exist_ok=True)
OUTBASE = os.path.join(RUNDIR, "carbonA_predictions"); WIN_ROOT = os.path.join(OUTBASE, "windows")
DIRS = {k: os.path.join(OUTBASE, k) for k in ("gff", "cds", "protein", "tracks", "log")}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
DEC = {"prefix": PREFIX, "gene_penalty": GENE_PENALTY, "intron_penalty": INTRON_PENALTY,
       "min_intron_length": MIN_INTRON_LENGTH, "genetic_code": CODE, "min_cds_length": MIN_CDS_LENGTH,
       "min_score": MIN_SCORE, "keep_partial": KEEP_PARTIAL,
       "splice_log_prior": {"GT-AG": 0.9935, "GC-AG": 0.0056, "AT-AC": 0.0009}}
dec_path = os.path.join(RUNDIR, "decode_config.json")
if os.path.exists(dec_path):
    dec_path = os.path.join(RUNDIR, f"decode_config_{datetime.now(NOW.tzinfo):%Y%m%d-%H%M%S}.json")
json.dump(DEC, open(dec_path, "w"), indent=2)
assert MIN_INTRON_LENGTH >= 4 and GENE_PENALTY >= 0 and INTRON_PENALTY >= 0

def gzip_to(src, dst_dir):
    dst = os.path.join(dst_dir, os.path.basename(src) + ".gz")
    with open(src, "rb") as fi, gzip.open(dst, "wb", compresslevel=6) as fo:
        shutil.copyfileobj(fi, fo)
    return dst

def wrap(s, n=60):
    return "\n".join(s[i:i + n] for i in range(0, len(s), n))

def rle(v):
    q = np.round(np.nan_to_num(v, nan=-1.0), 3).astype(np.float32)
    ch = np.flatnonzero(q[1:] != q[:-1]) + 1
    st = np.concatenate(([0], ch)); en = np.concatenate((ch, [len(q)])); val = q[st]
    keep = val >= 0
    return st[keep].astype(np.int64), en[keep].astype(np.int64), val[keep].astype(np.float64)

JOBS, DFAILED = [], []
for gi, (base, gpath) in enumerate(GENOMES, 1):
    print(f"\n===== [{gi}/{len(GENOMES)}] {base} =====")
    gdir = os.path.join(WIN_ROOT, base)
    if not os.path.exists(os.path.join(gdir, "DONE.json")):
        print("no complete predictions for this genome - run cell 7 first"); DFAILED.append(base); continue
    t0 = time.time()
    man = json.load(open(os.path.join(gdir, "manifest.json")))
    W, OV = man["key"]["window"], man["key"]["overlap"]
    # window index: record -> [(chunk file, offset into its logits, start, length)]
    by_rec = {}
    chunk_files = sorted(glob.glob(os.path.join(gdir, "chunk_*.npz")))
    for f in chunk_files:
        with np.load(f) as z:
            offs = np.concatenate(([0], np.cumsum(z["length"])[:-1]))
            for r, s, l, o in zip(z["rec"].tolist(), z["start"].tolist(), z["length"].tolist(), offs.tolist()):
                by_rec.setdefault(r, []).append((f, o, s, l))
    cache = {}
    def chunk(f):
        if f not in cache:
            if len(cache) >= 2:
                cache.pop(next(iter(cache)))
            with np.load(f) as z:
                cache[f] = z["logits"]
        return cache[f]
    prefix = PREFIX.strip().replace(" ", "_") or re.sub(r"[^A-Za-z0-9._-]+", "_", base)
    gff = os.path.join(PRED_LOCAL, f"{base}_gff.gff3")
    cds = os.path.join(PRED_LOCAL, f"{base}_cds.fasta")
    prot = os.path.join(PRED_LOCAL, f"{base}_protein.fasta")
    recs = [tuple(r) for r in man["records"]]
    header = [(s, n) for i, (s, n) in enumerate(recs) if i in by_rec]
    bws = {}
    if WRITE_BIGWIG and header:
        import pyBigWig
        for strand in ("plus", "minus"):
            bw = pyBigWig.open(os.path.join(PRED_LOCAL, f"{base}_cds_prob_{strand}.bw"), "w")
            bw.addHeader([(s, int(n)) for s, n in header], maxZooms=10)
            bws[strand] = bw
    n_genes = n_part = 0
    try:
        with open(gff, "w") as fg, open(cds, "w") as fc, open(prot, "w") as fp:
            fg.write("##gff-version 3\n")
            for s, n in header:
                fg.write(f"##sequence-region {s} 1 {n}\n")
            for idx, (h, seq) in enumerate(cc.read_fasta(gpath)):
                if idx not in by_rec:
                    continue
                seqid, L = recs[idx]
                assert len(seq) == L, f"{seqid}: length changed since prediction"
                wins = by_rec[idx]
                missing = set(cc.window_starts(L, W, OV)) - {w[2] for w in wins}
                assert not missing, f"{seqid}: {len(missing)} window(s) missing - re-run cell 7 (SKIP_DONE off) for {base}"
                P = cc.accumulate(L, ((s, chunk(f)[:, o:o + l]) for f, o, s, l in wins))
                genes = cc.annotate_record(seq, P[0], P[1], min_intron=MIN_INTRON_LENGTH,
                                           gene_penalty=GENE_PENALTY, intron_penalty=INTRON_PENALTY,
                                           genetic_code=CODE)
                keep = []
                for g in genes:
                    g["score"] = g["cds_mean"] * g["intron_mean"]
                    if g["cds_len"] < MIN_CDS_LENGTH or g["score"] < MIN_SCORE:
                        continue
                    if not KEEP_PARTIAL and (g["partial5"] or g["partial3"]):
                        continue
                    keep.append(g)
                lines = cc.gff_lines(seqid, keep, prefix)
                if lines:
                    fg.write("\n".join(lines) + "\n")
                for g in keep:
                    part = ",".join(x for x, f in (("5prime", g["partial5"]), ("3prime", g["partial3"])) if f)
                    desc = (f"{g['transcript_id']} gene={g['gene_id']} loc={seqid}:{g['exons'][0][0] + 1}-"
                            f"{g['exons'][-1][1]}({g['strand']}) exons={len(g['exons'])} score={g['score']:.3f}"
                            + (f" partial={part}" if part else ""))
                    fc.write(f">{desc}\n{wrap(g['cds'].decode())}\n")
                    aa = cc.translate_cds(g["cds"], g["offset"], CODE, g["partial3"])
                    fp.write(f">{desc}\n{wrap(aa)}\n")
                    n_part += bool(part)
                n_genes += len(keep)
                for strand, row in (("plus", 0), ("minus", 1)):
                    if strand in bws:
                        st, en, val = rle(P[row])
                        for a in range(0, len(st), 1_000_000):
                            bws[strand].addEntries([seqid] * len(st[a:a + 1_000_000]), st[a:a + 1_000_000],
                                                   ends=en[a:a + 1_000_000], values=val[a:a + 1_000_000])
                del P
        for bw in bws.values():
            bw.close()
        saved = [gzip_to(gff, DIRS["gff"]), gzip_to(cds, DIRS["cds"]), gzip_to(prot, DIRS["protein"])]
        for strand in bws:
            src = os.path.join(PRED_LOCAL, f"{base}_cds_prob_{strand}.bw")
            shutil.copy(src, DIRS["tracks"]); saved.append(os.path.join(DIRS["tracks"], os.path.basename(src)))
        print(f"{n_genes} gene models ({n_part} partial) in {(time.time() - t0) / 60:.1f} min ->",
              ", ".join(os.path.relpath(s, OUTBASE) for s in saved))
        JOBS.append({"base": base, "genome": gpath, "gff": gff, "cds": cds, "protein": prot})
    except Exception as e:
        for bw in bws.values():
            try:
                bw.close()
            except Exception:
                pass
        if STOP_ON_ERROR:
            raise
        print(f"ERROR: decoding failed on {base}: {type(e).__name__}: {e}"); DFAILED.append(base)

print(f"\nDone. {len(JOBS)} annotation(s) under {OUTBASE}/  (gff/ cds/ protein/ tracks/)")
if DFAILED:
    print(f"FAILED ({len(DFAILED)}):", ", ".join(DFAILED))
print("Decoder settings saved to", dec_path)

In [ ]:
#@title 9 · Summary of predictions { display-mode: "form" }
import os, gzip, json, statistics as st
from collections import Counter
import pandas as pd
from IPython.display import display

PRED_OUT = os.path.join(RUNDIR, "carbonA_predictions")

def _resolve(local, subdir):
    # prefer the local uncompressed file; fall back to the gzipped copy on Drive
    if os.path.exists(local):
        return local
    gz = os.path.join(PRED_OUT, subdir, os.path.basename(local) + ".gz")
    return gz if os.path.exists(gz) else local

def _open(p):
    return gzip.open(p, "rt") if p.endswith(".gz") else open(p)

def gff_stats(path):
    feat = Counter(); cds_bp = 0; partial = 0; scores = []
    if not os.path.exists(path):
        return feat, cds_bp, partial, scores
    with _open(path) as fh:
        for ln in fh:
            if ln.startswith("#"):
                continue
            f = ln.rstrip("\n").split("\t")
            if len(f) < 9:
                continue
            feat[f[2].lower()] += 1
            if f[2] == "CDS":
                cds_bp += int(f[4]) - int(f[3]) + 1
            elif f[2] == "gene":
                scores.append(float(f[5]))
                partial += "partial=" in f[8]
    return feat, cds_bp, partial, scores

def prot_lengths(fa):
    L = []; cur = 0
    if not os.path.exists(fa):
        return L
    with _open(fa) as fh:
        for ln in fh:
            if ln.startswith(">"):
                if cur:
                    L.append(cur)
                cur = 0
            else:
                cur += sum(1 for ch in ln.strip() if ch not in "*.")
    if cur:
        L.append(cur)
    return L

rows = []
for j in JOBS:
    feat, cds_bp, partial, scores = gff_stats(_resolve(j["gff"], "gff"))
    L = prot_lengths(_resolve(j["protein"], "protein"))
    gbp = sum(n for _, n in GENOME_INFO[j["base"]])
    mrna = feat.get("mrna", 0)
    rows.append({
        "genome": j["base"],
        "genome_Mb": round(gbp / 1e6, 1),
        "genes": feat.get("gene", 0),
        "genes/Mb": round(feat.get("gene", 0) / (gbp / 1e6), 1) if gbp else 0,
        "partial": partial,
        "proteins": len(L),
        "CDS": feat.get("cds", 0),
        "CDS/mRNA": round(feat.get("cds", 0) / mrna, 2) if mrna else 0,
        "mean_aa": round(st.mean(L)) if L else 0,
        "median_aa": round(st.median(L)) if L else 0,
        "max_aa": max(L) if L else 0,
        "total_CDS_Mb": round(cds_bp / 1e6, 2),
        "median_score": round(st.median(scores), 3) if scores else 0,
        "score>=0.5": round(sum(s >= 0.5 for s in scores) / len(scores) * 100, 1) if scores else 0,
    })

df = pd.DataFrame(rows)
out_tsv = os.path.join(RUNDIR, "prediction_summary.tsv")
df.to_csv(out_tsv, sep="\t", index=False)
display(df)
print("Saved:", out_tsv)
print("Outputs under:", PRED_OUT, " (gff/ cds/ protein/ tracks/ log/ windows/)")
print("`score>=0.5` = % of gene models with score >= 0.5 (column 6 of the GFF3).")

## BUSCO completeness

Two independent checks, each toggled in the next cell:
- **proteins mode** — runs on Carbon-A's predicted proteins; how complete the **annotation** is. HMMER only -> fast.
- **genome mode** — runs on each assembly via miniprot; the assembly's completeness **ceiling**, independent of annotation. Needs bbtools + miniprot and is **much slower** (CPU-bound — for whole genomes prefer HPC).

Pick a lineage matching the clade of your genomes, or choose **other** and type any dataset name. List all with `busco --list-datasets`. Every genome in the batch is scored against the same lineage.

In [ ]:
#@title 10 · BUSCO options { display-mode: "form" }
RUN_PROTEIN_BUSCO = True   #@param {type:"boolean"}
RUN_GENOME_BUSCO = False   #@param {type:"boolean"}
BUSCO_LINEAGE = "eudicotyledons_odb12.2"  #@param ["eukaryota_odb12.2", "viridiplantae_odb12.2", "embryophyta_odb12.2", "eudicotyledons_odb12.2", "liliopsida_odb12.2", "chlorophyta_odb12.2", "stramenopiles_odb12.2", "oomycota_odb12.2", "fungi_odb12.2", "ascomycota_odb12.2", "basidiomycota_odb12.2", "sordariomycetes_odb12.2", "saccharomycetes_odb12.2", "metazoa_odb12.2", "arthropoda_odb12.2", "insecta_odb12.2", "lepidoptera_odb12.2", "nematoda_odb12.2", "mollusca_odb12.2", "vertebrata_odb12.2", "mammalia_odb12.2", "aves_odb12.2", "actinopterygii_odb12.2", "other"] {allow-input: true}
#@markdown If `BUSCO_LINEAGE` is **other**, type the dataset name here:
BUSCO_LINEAGE_CUSTOM = ""  #@param {type:"string"}

LINEAGE_BUSCO = BUSCO_LINEAGE_CUSTOM.strip() if BUSCO_LINEAGE == "other" else BUSCO_LINEAGE
assert LINEAGE_BUSCO and LINEAGE_BUSCO != "other", "Choose a lineage, or set BUSCO_LINEAGE_CUSTOM when using 'other'."
print("lineage:", LINEAGE_BUSCO)

In [ ]:
#@title 11 · Install BUSCO (+ genome-mode tools if enabled) { display-mode: "form" }
#@markdown HMMER + BUSCO for both modes. Genome mode also builds miniprot and installs bbtools. ~1-3 min.
import os, subprocess
subprocess.call("apt-get -qq install -y hmmer > /dev/null", shell=True)
if not os.path.isdir("/content/busco"):
    subprocess.call("git clone -q --depth 1 https://gitlab.com/ezlab/busco.git /content/busco", shell=True)
subprocess.call("pip install -q /content/busco biopython pandas requests scipy", shell=True)

if RUN_GENOME_BUSCO:
    subprocess.call("apt-get -qq install -y bbmap > /dev/null", shell=True)
    subprocess.call("ln -sf /usr/share/bbmap/stats.sh /usr/bin/stats.sh", shell=True)
    subprocess.call("ln -sf /usr/share/bbmap/statswrapper.sh /usr/bin/statswrapper.sh", shell=True)
    if not os.path.exists("/usr/local/bin/miniprot"):
        subprocess.call("git clone -q --depth 1 https://github.com/lh3/miniprot /content/miniprot", shell=True)
        subprocess.call("make -C /content/miniprot -j2 && cp /content/miniprot/miniprot /usr/local/bin/", shell=True)
    print("miniprot:", subprocess.run(["miniprot", "--version"], capture_output=True, text=True).stdout.strip())
print("busco   :", subprocess.run(["busco", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 12 · Run BUSCO + compile results table { display-mode: "form" }
import os, glob, re, csv, gzip, shutil, subprocess
BUSCO_LOCAL = "/content/busco_runs"; os.makedirs(BUSCO_LOCAL, exist_ok=True)
assert RUN_PROTEIN_BUSCO or RUN_GENOME_BUSCO, "Both BUSCO toggles are off (cell 10)."

LIN_TAG = re.sub(r"[^A-Za-z0-9._-]+", "_", LINEAGE_BUSCO)      # goes into every output name

def parse_summary(path):
    m = re.search(r"C:([\d.]+)%\[S:([\d.]+)%,D:([\d.]+)%\],F:([\d.]+)%,M:([\d.]+)%,n:(\d+)", open(path).read())
    keys = ["C%", "S%", "D%", "F%", "M%", "n"]
    return dict(zip(keys, m.groups())) if m else {k: "" for k in keys}

def local_plain(path, subdir):
    # the plain FASTA, restored from the gzipped Drive copy after a runtime reset
    if not os.path.exists(path):
        gz = os.path.join(RUNDIR, "carbonA_predictions", subdir, os.path.basename(path) + ".gz")
        if os.path.exists(gz):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            with gzip.open(gz, "rb") as fi, open(path, "wb") as fo:
                shutil.copyfileobj(fi, fo)
    return path

def local_genome(j):
    # BUSCO genome mode wants a plain FASTA
    if not j["genome"].endswith(".gz"):
        return j["genome"]
    out = os.path.join("/content/run/genomes_plain", j["base"] + ".fasta")
    if not os.path.exists(out):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        with gzip.open(j["genome"], "rb") as fi, open(out, "wb") as fo:
            shutil.copyfileobj(fi, fo)
    return out

rows = []
BUSCO_RUNS = []          # directory names produced by this cell, zipped by cell 13
for j in JOBS:
    targets = []
    if RUN_PROTEIN_BUSCO:
        prot = local_plain(j["protein"], "protein")
        assert os.path.exists(prot), f"No protein FASTA for {j['base']} - run cell 8."
        targets.append(("proteins", prot, ["-m", "proteins"], f"{j['base']}_busco_prot_{LIN_TAG}"))
    if RUN_GENOME_BUSCO:
        targets.append(("genome", local_genome(j), ["-m", "genome", "--miniprot"], f"{j['base']}_busco_genome_{LIN_TAG}"))
    for mode, inp, modeargs, name in targets:
        cmd = ["busco", "-i", inp, *modeargs, "-l", LINEAGE_BUSCO, "-o", name,
               "--out_path", BUSCO_LOCAL, "-c", str(os.cpu_count() or 2), "-f"]
        print(f"\n##### BUSCO {mode}: {j['base']} #####")
        subprocess.run(cmd, check=True)
        summ = glob.glob(os.path.join(BUSCO_LOCAL, name, "short_summary*.txt"))
        rows.append({"genome": j["base"], "mode": mode, "lineage": LINEAGE_BUSCO,
                     **(parse_summary(summ[0]) if summ else {})})
        BUSCO_RUNS.append(name)

cols = ["genome", "mode", "lineage", "C%", "S%", "D%", "F%", "M%", "n"]
BUSCO_OUT = os.path.join(RUNDIR, "carbonA_predictions", "busco")
os.makedirs(BUSCO_OUT, exist_ok=True)
table = os.path.join(BUSCO_OUT, "busco_summary.tsv")

# one table per run folder, accumulating across lineages: rows are keyed by
# (genome, mode, lineage), so re-running a lineage replaces its rows instead of duplicating them
key = lambda r: (r["genome"], r["mode"], r["lineage"])
merged = {}
if os.path.exists(table):
    with open(table) as f:
        for r in csv.DictReader(f, delimiter="\t"):
            merged[key(r)] = r
for r in rows:
    merged[key(r)] = r
with open(table, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, delimiter="\t"); w.writeheader()
    w.writerows([merged[k] for k in sorted(merged)])

print("\n=== BUSCO summary table ===")
print("\t".join(cols))
for k in sorted(merged):
    print("\t".join(str(merged[k].get(c, "")) for c in cols))
print(f"\nTable saved: {table}  ({len(merged)} row(s), {len({k[2] for k in merged})} lineage(s))")

In [ ]:
#@title 13 · Compress & save BUSCO run directories { display-mode: "form" }
import os, glob, shutil
BUSCO_LOCAL = "/content/busco_runs"
BUSCO_OUT = os.path.join(RUNDIR, "carbonA_predictions", "busco"); os.makedirs(BUSCO_OUT, exist_ok=True)

# only the runs cell 12 just produced, so earlier lineages in this session are not re-zipped
todo = [os.path.join(BUSCO_LOCAL, n) for n in BUSCO_RUNS] if "BUSCO_RUNS" in dir() \
       else sorted(glob.glob(os.path.join(BUSCO_LOCAL, "*")))
made = []
for d in todo:
    if os.path.isdir(d):
        arc = shutil.make_archive(os.path.join(BUSCO_OUT, os.path.basename(d)), "zip", d)
        made.append(arc); print("zipped:", os.path.basename(arc))
if not MOUNT_DRIVE:
    from google.colab import files
    for z in made:
        files.download(z)
print("\nBUSCO zips and busco_summary.tsv in:", BUSCO_OUT)
print("Zips present:", len(glob.glob(os.path.join(BUSCO_OUT, "*.zip"))))

---
### Notes & troubleshooting
- **Outputs** land in a timestamped run folder, `WORKDIR/runs/run_<date>-<time>[_tag]/`: everything in `carbonA_predictions/` — `gff/`, `cds/`, `protein/` (all gzipped), `tracks/` (bigWig CDS probability per strand), `windows/` (the raw window predictions cell 8 decodes; int8 logits, ~1 byte per bp), `log/` and `busco/` (one zip per BUSCO run plus `busco_summary.tsv`); alongside them `prediction_summary.tsv`, `run_config.json` (model commit, window/overlap, precision, GPU, package versions) and `decode_config.json` (decoder settings; a re-decode adds a timestamped copy). Weights live outside the run folders in `WORKDIR/carbonA_model/<commit>/` and are downloaded once.
- **Starting a new run:** re-run cell 3 — it stamps a fresh folder. Re-running cells 7-13 on their own keeps writing into the current one.
- **Resuming after a disconnect:** in the new session set `RESUME_RUN` in cell 3 to the interrupted run folder's name and run cells 1-7 again. Finished genomes are skipped and the interrupted one continues from its last checkpoint (`CHECKPOINT_MINUTES`). The window/overlap settings, model commit and FASTA must be unchanged — cell 7 checks.
- **Re-decoding:** cell 8 only reads the saved windows, so it can be re-run with other penalties, genetic code or filters in seconds to minutes; each re-decode overwrites the GFF3/FASTA/tracks and records its settings. In a new session no GPU is needed: run cells 1-5 (with `RESUME_RUN`), then 8.
- **What the decoder does:** per strand, a Viterbi path through intergenic / start codon / exon (codon position x stop-relevant prefix) / intron (phase x splice class) states. A gene starts at ATG (never split by an intron) and ends at the first in-frame stop for the chosen genetic code, including stops spanning a splice junction; introns are GT..AG, GC..AG or AT..AC with log-priors 0.9935 / 0.0056 / 0.0009 and at least `MIN_INTRON_LENGTH` bp. Emissions are log *p* (CDS) and log(1 − *p*) (intergenic, intron), exactly as in the report; `GENE_PENALTY` / `INTRON_PENALTY` replace its transition network. Bases with an ambiguous nucleotide cannot be coding, so a gene is cut at an `N`.
- **Partial genes:** a sequence may start or end inside a gene; such models are kept (`KEEP_PARTIAL`) and flagged `partial=5prime` / `3prime`. CDS phases and protein translations account for the missing codon positions.
- **Gene score** (GFF3 column 6, also `mean_cds_p` / `mean_intron_q` attributes) is a simple consistency measure, not the report's trained confidence score: well-supported genes sit near 1. Filter with `MIN_SCORE` or downstream.
- **Overlapping genes on opposite strands** are kept (the strands are decoded independently); on the same strand only one gene can occupy a position, and only one isoform per locus is reported.
- **Soft-masked genomes need no preprocessing** — sequences are upper-cased before tokenization (lowercase bases would otherwise not form 6-mers). Do not hardmask: `N` cannot be coding.
- **Speed:** Carbon-A is a 1.2 B-parameter transformer over 16,384-token windows. Cell 6 reports the measured time per window and an estimate for your batch. bf16 GPUs (A100 / L4 / H100) are much faster than a T4. `WINDOW_OVERLAP = 49152` doubles GPU time for slightly smoother window joins.
- **GPU OOM:** the batch is halved automatically; with batch 1 a full window fits in ~6 GB, so OOM at batch 1 means another process holds GPU memory (restart the runtime).
- **Very large chromosomes** need host RAM: ~17 bytes per bp while decoding (a 500 Mb chromosome ~8.5 GB). Use a high-RAM runtime for giant chromosomes.
- **Precision:** bf16 matches the model's own inference checks; fp16 (T4) is compared with fp32 in cell 6. Results can differ slightly between GPU types and precisions.
- **BUSCO + Drive:** BUSCO uses symlinks (unsupported on the Drive mount), so it always runs locally (`/content/busco_runs`); cell 13 zips each run into `carbonA_predictions/busco/`, next to `busco_summary.tsv`. Change `BUSCO_LINEAGE` and re-run cells 10-13 to add another lineage to the same table.
- **Genome BUSCO is slow** on Colab cores — fine for scaffolds, better on HPC for whole genomes.
- **Edit a hidden cell:** menu (vertical dots) > *Show code*.
- Cite: the Carbon-A technical report and model card (HuggingFaceBio, 2026; [model](https://huggingface.co/HuggingFaceBio/Carbon-A-1.2B)); GENERanno (the source annotator); Manni *et al.* (2021) *Mol Biol Evol* for BUSCO.